In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggdist)
library(scales)

csv_path <- "ladder_validation_outputs_final/AMLScaffoldcompare_validation_results_final_2methods.csv"
run_label <- "AML — Scaffold vs Paragraph"

#keep_models <- c("BioLORD-2023", "BioSentVec", "BiomedBERT", "MedCPT", "PubMedBERT")
#keep_models <- c("BioLORD-2023", "MedCPT")
keep_models <- c("BioLORD-2023")
method_pal <- c(
  "Scaffolded Prompting"            = "#ca0020",
  "Free Form Analytical Prompting"  = "#0571b0"
)

theme_nature <- function(base_size = 10) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
  theme(
    panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
    panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
    panel.grid.minor   = element_blank(),
    axis.line          = element_blank(),
    axis.ticks         = element_line(colour = "black", linewidth = 0.45),
    axis.ticks.length  = unit(3, "pt"),
    axis.title         = element_text(face = "bold", size = base_size),
    axis.text          = element_text(colour = "black", size = base_size - 1),
    axis.text.x        = element_text(angle = 30, hjust = 1),
    legend.title       = element_text(face = "bold", size = base_size - 1),
    legend.text        = element_text(size = base_size - 2),
    legend.key         = element_blank(),
    legend.background  = element_blank(),
    plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
    plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
    strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
    strip.text         = element_text(face = "bold", size = base_size - 1),
    plot.margin        = margin(8, 12, 8, 8)
  )
}

# ============================================================
# LOAD & PREPARE DATA
# ============================================================
results <- read.csv(csv_path, stringsAsFactors = FALSE) %>%
  filter(Model %in% keep_models) %>%
  mutate(
    Model  = factor(Model, levels = keep_models),
    Method = recode(Method,
                    "Final Scaffold"  = "Scaffolded Prompting",
                    "Final Paragraph" = "Free Form Analytical Prompting"),
    Method = factor(Method, levels = c("Scaffolded Prompting",
                                       "Free Form Analytical Prompting"))
  )

# ============================================================
# PANEL A — Win counts
# ============================================================
wins <- results %>%
  filter(Winner == "True") %>%
  count(Model, Method)

pA <- ggplot(wins, aes(x = Model, y = n, fill = Method)) +
  geom_col(position = position_dodge(width = 0.72), width = 0.65,
           colour = "white", linewidth = 0.4) +
  geom_text(aes(label = n),
            position = position_dodge(width = 0.72),
            vjust = -0.4, size = 2.6, fontface = "bold", colour = "black") +
  scale_fill_manual(values = method_pal, drop = FALSE) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15)),
                     breaks = pretty_breaks(4)) +
  labs(
    title    = "a",
    subtitle = "Win counts by embedding model",
    x        = "Embedding Model",
    y        = "Number of Wins",
    fill     = "Method"
  ) +
  theme_nature() +
  theme(panel.grid.major.x = element_blank())

# ============================================================
# PANEL B — Cosine similarity raincloud
# ============================================================
pB <- ggplot(results, aes(x = Model, y = Similarity,
                           fill = Method, colour = Method)) +
  stat_halfeye(adjust = 0.8, width = 0.45, .width = 0,
               justification = -0.2, point_colour = NA, alpha = 0.72,
               position = position_dodge(width = 0.7)) +
  geom_boxplot(outlier.shape = NA, width = 0.14, linewidth = 0.45,
               colour = "black", alpha = 0.55,
               position = position_dodge(width = 0.7)) +
  geom_point(aes(x = as.numeric(Model) + 0.0),
             position = position_jitterdodge(jitter.width = 0.05,
                                             dodge.width = 0.7,
                                             seed = 42),
             size = 0.8, alpha = 0.3, shape = 16) +
  scale_fill_manual(values = method_pal, drop = FALSE) +
  scale_colour_manual(values = method_pal, drop = FALSE) +
  scale_y_continuous(breaks = seq(0, 1, 0.2),
                     limits = c(NA, 1.05)) +
  labs(
    title    = "b",
    subtitle = "Cosine similarity distribution",
    x        = "Embedding Model",
    y        = "Cosine Similarity",
    fill     = "Method",
    colour   = "Method"
  ) +
  theme_nature() +
  guides(fill = guide_legend(title = "Method"),
         colour = guide_legend(title = "Method"))

# ============================================================
# ASSEMBLE & SAVE
# ============================================================
fig <- (pA | pB) +
  plot_annotation(
    title  = run_label,
    theme  = theme(
      plot.title = element_text(face = "bold", size = 11, hjust = 0,
                                family = "Helvetica")
    )
  ) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom",
        legend.direction = "horizontal")

ggsave("Fig_LADDER_ScaffoldVsParagraphBiolord.pdf",
       fig, width = 14, height = 5.5, dpi = 300)
ggsave("Fig_LADDER_ScaffoldVsParagraphBiolord.png",
       fig, width = 14, height = 5.5, dpi = 300)

cat("Saved Fig_LADDER_ScaffoldVsParagraphBiolord.pdf/.png\n")